# AM5061 · Week 1 · The 28 kW dairy heat pump

**Design of Thermal and Fluid Systems** · Applied Mechanics, IIT Madras · Jul–Nov 2026

Run the two setup cells below once, then work down the notebook. Nothing needs to be installed on your own machine.


## Setup

Run these two cells first. The second one writes the course helper module, so this notebook is self-contained.


In [ ]:
#@title Install the property library  { display-mode: "form" }
!pip install -q CoolProp openpyxl
print('CoolProp ready')


In [ ]:
%%writefile am5061.py
"""AM5061 - Design of Thermal and Fluid Systems.
Shared helpers for the course notebooks.

Design of Thermal and Fluid Systems, IIT Madras, Jul-Nov 2026.

This module is deliberately thin. It wraps CoolProp with names and units that
match the lecture notation, adds an Excel writer that produces workbooks you
can actually filter, and sets a consistent plot style. It does NOT hide the
engineering: every case notebook still writes its own equations.

Install (first cell of any Colab notebook):
    !pip install -q CoolProp openpyxl

Units are SI throughout, with ONE exception that is flagged everywhere it
appears: temperatures in function arguments named `..._C` are in Celsius,
because that is how the case briefs state them. Everything internal is kelvin.
"""
from __future__ import annotations

import math

from CoolProp.CoolProp import PropsSI, PhaseSI

__all__ = [
    "K", "C", "State", "state", "sat_liquid", "sat_vapour", "p_sat", "T_sat",
    "h_fg", "critical", "fluids", "solve", "sweep", "to_excel",
    "style_plots", "NAVY", "ORANGE", "BLUE", "MUTED",
]

# ---------------------------------------------------------------- constants
NAVY, ORANGE, BLUE, MUTED = "#1F3864", "#ED7D31", "#4472C4", "#59626E"
T0 = 273.15


def K(t_celsius: float) -> float:
    """Celsius -> kelvin. Use this at the boundary, never inside a formula."""
    return t_celsius + T0


def C(t_kelvin: float) -> float:
    """Kelvin -> Celsius, for reporting only."""
    return t_kelvin - T0


# ------------------------------------------------------------------- states
class State:
    """A thermodynamic state. Immutable, and it knows its own fluid.

    Construct it with any two independent properties:
        State("R134a", P=1e6, T=K(70))
        State("Water", P=101325, Q=0)      # saturated liquid
        State("R134a", P=p_cond, H=h2)

    Then read properties as attributes: .T .p .h .s .d .cp .x
    Attribute names match the lecture notation, not CoolProp's letter codes,
    so a student reading the notebook does not need the CoolProp manual open.
    """

    _MAP = {"T": "T", "P": "P", "H": "H", "S": "S", "D": "D", "Q": "Q"}

    def __init__(self, fluid: str, **kw):
        if len(kw) != 2:
            raise ValueError(
                f"a state needs exactly two properties, got {list(kw)}. "
                "Two and only two - that is the phase rule, not a quirk."
            )
        (n1, v1), (n2, v2) = kw.items()
        for n in (n1, n2):
            if n not in self._MAP:
                raise ValueError(f"unknown property {n!r}; use T, P, H, S, D or Q")
        self.fluid, self._args = fluid, (n1, v1, n2, v2)

    def _get(self, what: str) -> float:
        n1, v1, n2, v2 = self._args
        return PropsSI(what, n1, v1, n2, v2, self.fluid)

    # Named so they read like the equations on the slides.
    T  = property(lambda s: s._get("T"),  doc="temperature, K")
    p  = property(lambda s: s._get("P"),  doc="pressure, Pa")
    h  = property(lambda s: s._get("H"),  doc="specific enthalpy, J/kg")
    s  = property(lambda s: s._get("S"),  doc="specific entropy, J/kg.K")
    d  = property(lambda s: s._get("D"),  doc="density, kg/m3")
    cp = property(lambda s: s._get("C"),  doc="cp, J/kg.K")
    mu = property(lambda s: s._get("V"),  doc="dynamic viscosity, Pa.s")
    k  = property(lambda s: s._get("L"),  doc="thermal conductivity, W/m.K")
    x  = property(lambda s: s._get("Q"),  doc="vapour quality, - (=-1 if single phase)")

    @property
    def T_C(self) -> float:
        return C(self.T)

    @property
    def phase(self) -> str:
        n1, v1, n2, v2 = self._args
        return PhaseSI(n1, v1, n2, v2, self.fluid)

    def __repr__(self):
        try:
            return (f"State({self.fluid}: {self.T_C:.2f} C, {self.p/1e5:.3f} bar, "
                    f"h={self.h/1e3:.2f} kJ/kg, {self.phase})")
        except Exception:
            return f"State({self.fluid}, {self._args})"


def state(fluid: str, **kw) -> State:
    """Shorthand for State(...)."""
    return State(fluid, **kw)


def sat_liquid(fluid: str, *, T=None, p=None) -> State:
    """Saturated liquid at T or p. Give one, not both."""
    if (T is None) == (p is None):
        raise ValueError("give exactly one of T or p")
    return State(fluid, T=T, Q=0) if T is not None else State(fluid, P=p, Q=0)


def sat_vapour(fluid: str, *, T=None, p=None) -> State:
    """Saturated vapour at T or p."""
    if (T is None) == (p is None):
        raise ValueError("give exactly one of T or p")
    return State(fluid, T=T, Q=1) if T is not None else State(fluid, P=p, Q=1)


def p_sat(fluid: str, T: float) -> float:
    """Saturation pressure, Pa. For a BLEND this is the bubble-point pressure."""
    return PropsSI("P", "T", T, "Q", 0, fluid)


def T_sat(fluid: str, p: float) -> float:
    """Saturation temperature, K.

    WARNING for blends: a zeotropic mixture has no single saturation
    temperature. This returns the BUBBLE point. Use glide() to see the spread.
    """
    return PropsSI("T", "P", p, "Q", 0, fluid)


def glide(fluid: str, p: float) -> float:
    """Dew minus bubble temperature at p, K. Zero for a pure fluid."""
    return (PropsSI("T", "P", p, "Q", 1, fluid)
            - PropsSI("T", "P", p, "Q", 0, fluid))


def h_fg(fluid: str, *, T=None, p=None) -> float:
    """Latent heat, J/kg."""
    return sat_vapour(fluid, T=T, p=p).h - sat_liquid(fluid, T=T, p=p).h


def critical(fluid: str) -> dict:
    """Critical point, for checking you are not extrapolating past it."""
    return {"T": PropsSI("TCRIT", fluid), "p": PropsSI("PCRIT", fluid)}


def fluids() -> list:
    """Every fluid CoolProp knows. There are about 130."""
    import CoolProp
    return sorted(CoolProp.__fluids__)


# ------------------------------------------------------------------ solvers
def solve(f, x0, *, tol=1e-10, max_iter=200, bracket=None):
    """Find x where f(x) = 0.

    Uses Brent's method when you give a bracket (robust, always converges if
    the bracket is valid), otherwise secant from x0. Raises with a readable
    message rather than returning a wrong answer silently, which is the whole
    problem with doing this in a spreadsheet.
    """
    from scipy.optimize import brentq, newton
    if bracket is not None:
        a, b = bracket
        fa, fb = f(a), f(b)
        if fa * fb > 0:
            raise ValueError(
                f"f({a:g})={fa:g} and f({b:g})={fb:g} have the same sign, so no "
                "root is bracketed. Widen the bracket or check the equation."
            )
        return brentq(f, a, b, xtol=tol, maxiter=max_iter)
    return newton(f, x0, tol=tol, maxiter=max_iter)


def sweep(fn, values, *, name="x"):
    """Run fn(v) for each v and collect the results as a list of dicts.

    fn must return a dict. The sweep variable is added under `name`, so the
    result drops straight into to_excel().
    """
    rows = []
    for v in values:
        out = fn(v)
        if not isinstance(out, dict):
            raise TypeError("the swept function must return a dict of results")
        rows.append({name: v, **out})
    return rows


# -------------------------------------------------------------------- excel
def to_excel(path, sheets: dict, *, sources=None, summary=None, title=None):
    """Write a workbook that is genuinely usable.

    Every numeric cell is written as a number (not text), AutoFilter is on,
    the header row is frozen, and columns are sized to content. A Summary and
    a Sources sheet are always present, because a result you cannot trace is
    not an engineering deliverable.

        sheets  = {"Sweep": [ {...}, {...} ], ...}   list of dicts per sheet
        sources = [ ("what", "where it came from"), ... ]
        summary = [ ("quantity", value, "units"), ... ]
    """
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

    wb = Workbook()
    wb.remove(wb.active)
    head_font = Font(bold=True, color="FFFFFF", name="Calibri")
    head_fill = PatternFill("solid", fgColor="1F3864")

    def _write(ws, rows, headers=None):
        headers = headers or (list(rows[0].keys()) if rows else [])
        for j, hname in enumerate(headers, 1):
            c = ws.cell(row=1, column=j, value=hname)
            c.font, c.fill = head_font, head_fill
            c.alignment = Alignment(horizontal="left")
        for i, row in enumerate(rows, 2):
            for j, hname in enumerate(headers, 1):
                v = row.get(hname)
                # Numbers stay numbers. This is the single most common way a
                # delivered workbook turns out not to be filterable.
                if isinstance(v, bool):
                    v = str(v)
                elif isinstance(v, (int, float)) and not isinstance(v, bool):
                    v = float(v) if isinstance(v, float) else v
                ws.cell(row=i, column=j, value=v)
        if rows:
            ws.auto_filter.ref = (f"A1:{get_column_letter(len(headers))}"
                                  f"{len(rows) + 1}")
        ws.freeze_panes = "A2"
        for j, hname in enumerate(headers, 1):
            width = max([len(str(hname))] +
                        [len(f"{r.get(hname)}") for r in rows[:200]]) + 3
            ws.column_dimensions[get_column_letter(j)].width = min(width, 42)

    # Summary first, so it is what opens.
    ws = wb.create_sheet("Summary")
    ws["A1"] = title or "AM5061 results"
    ws["A1"].font = Font(bold=True, size=14, color="1F3864")
    r = 3
    for item in (summary or []):
        for j, v in enumerate(item, 1):
            ws.cell(row=r, column=j, value=v)
        r += 1
    ws.column_dimensions["A"].width = 46
    ws.column_dimensions["B"].width = 18
    ws.column_dimensions["C"].width = 14

    for sname, rows in sheets.items():
        _write(wb.create_sheet(sname[:31]), rows)

    ws = wb.create_sheet("Sources")
    _write(ws, [{"item": a, "source": b} for a, b in (sources or [])])

    wb.save(path)
    return path


# --------------------------------------------------------------- plot style
def style_plots():
    """Match the lecture decks, so figures in a report look like the slides."""
    import matplotlib as mpl
    mpl.rcParams.update({
        "figure.figsize": (7.2, 4.4), "figure.dpi": 110,
        "axes.edgecolor": MUTED, "axes.labelcolor": NAVY,
        "axes.titlecolor": NAVY, "axes.titlesize": 11.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
        "xtick.color": MUTED, "ytick.color": MUTED,
        "font.size": 10, "legend.frameon": False,
        "axes.prop_cycle": mpl.cycler(color=[NAVY, ORANGE, BLUE, "#7F9DB9"]),
    })


---
## The case

A dairy needs **28 kW of heating** to pasteurise milk. You are specifying a
vapour-compression heat pump on **R134a**, condensing at **70 °C** and
evaporating at **10 °C**.

Deliverable **D-1**: the four state points, the COP, the mass flow and the
compressor power. Then answer the design question at the bottom.

### What you are actually doing

There are no components here. This is **sixteen equations in four state
points**. The cycle diagram is a drawing that tells you which enthalpy is
which, nothing more. Real components with ports arrive in Week 7.


## 1. The design specification

Everything the brief gives you, in one place. Change these and the whole
notebook re-runs.


In [ ]:
import am5061 as am
from CoolProp.CoolProp import PropsSI
import numpy as np, matplotlib.pyplot as plt
am.style_plots()

FLUID  = "R134a"
Q_H    = 28e3          # W      required heating duty
T_cond = am.K(70)      # K      condensing temperature
T_evap = am.K(10)      # K      evaporating temperature
eta_s  = 1.0           # -      isentropic efficiency (1.0 = ideal, for now)
dT_sub = 0.0           # K      condenser subcooling
dT_sup = 0.0           # K      evaporator superheat

print(f"critical point of {FLUID}: "
      f"{am.C(am.critical(FLUID)['T']):.2f} C, {am.critical(FLUID)['p']/1e5:.2f} bar")
print(f"condensing at {am.C(T_cond):.0f} C - comfortably subcritical, so the "
      "cycle has a two-phase condenser.")


## 2. The two pressures

The evaporator and condenser each sit at the saturation pressure of their
temperature. This is the only place the fluid choice enters the *structure* of
the problem, and it is why a fluid with a sensible pressure ratio over your
temperature lift is the first thing you look for.


In [ ]:
p_evap = am.p_sat(FLUID, T_evap)
p_cond = am.p_sat(FLUID, T_cond)

print(f"p_evap = {p_evap/1e5:7.4f} bar   at {am.C(T_evap):.0f} C")
print(f"p_cond = {p_cond/1e5:7.4f} bar   at {am.C(T_cond):.0f} C")
print(f"pressure ratio = {p_cond/p_evap:.3f}")


## 3. The four state points

| | where | how it is fixed |
|---|---|---|
| 1 | compressor suction | saturated vapour at `p_evap`, plus superheat |
| 2 | compressor discharge | at `p_cond`, from the isentropic state and `eta_s` |
| 3 | condenser outlet | saturated liquid at `p_cond`, less subcooling |
| 4 | evaporator inlet | isenthalpic expansion, so `h4 = h3` |

State 2 is the only one needing care. Compression is *isentropic* to the ideal
point 2s, then the real work is scaled by the isentropic efficiency.


In [ ]:
# --- 1: compressor suction -------------------------------------------
if dT_sup > 0:
    st1 = am.State(FLUID, P=p_evap, T=am.T_sat(FLUID, p_evap) + dT_sup)
else:
    st1 = am.sat_vapour(FLUID, p=p_evap)
h1, s1 = st1.h, st1.s

# --- 2s: isentropic discharge, then 2: real discharge -----------------
h2s = PropsSI("H", "P", p_cond, "S", s1, FLUID)
h2  = h1 + (h2s - h1) / eta_s
st2 = am.State(FLUID, P=p_cond, H=h2)

# --- 3: condenser outlet ----------------------------------------------
if dT_sub > 0:
    st3 = am.State(FLUID, P=p_cond, T=am.T_sat(FLUID, p_cond) - dT_sub)
else:
    st3 = am.sat_liquid(FLUID, p=p_cond)
h3 = st3.h

# --- 4: isenthalpic expansion -----------------------------------------
h4  = h3
st4 = am.State(FLUID, P=p_evap, H=h4)

print(f"{'pt':>3} {'p, bar':>9} {'T, C':>9} {'h, kJ/kg':>10} {'x, -':>8}  phase")
for n, st in ((1, st1), (2, st2), (3, st3), (4, st4)):
    x = st.x
    xs = f"{x:8.4f}" if 0 <= x <= 1 else "       -"
    # CoolProp calls a point ON the dome "twophase". True, but it reads oddly
    # for a saturated vapour, so name the two edges explicitly.
    ph = ("sat. liquid" if x == 0 else "sat. vapour" if x == 1 else st.phase)
    print(f"{n:>3} {st.p/1e5:9.4f} {st.T_C:9.3f} {st.h/1e3:10.3f} {xs}  {ph}")


## 4. Performance

Specific quantities first, then scale up to the duty the dairy asked for.


In [ ]:
w_in = h2 - h1          # J/kg   compressor work
q_H  = h2 - h3          # J/kg   heat rejected in the condenser
q_L  = h1 - h4          # J/kg   refrigeration effect

COP_actual = q_H / w_in
COP_carnot = T_cond / (T_cond - T_evap)
eta_II     = COP_actual / COP_carnot

m_flow = Q_H / q_H
W_comp = m_flow * w_in
Q_L    = m_flow * q_L

print(f"  w_in        {w_in/1e3:10.3f} kJ/kg")
print(f"  q_H         {q_H/1e3:10.3f} kJ/kg")
print(f"  q_L         {q_L/1e3:10.3f} kJ/kg")
print(f"  COP_actual  {COP_actual:10.5f}")
print(f"  COP_carnot  {COP_carnot:10.5f}")
print(f"  eta_II      {eta_II:10.5f}")
print(f"  m_flow      {m_flow:10.5f} kg/s")
print(f"  W_comp      {W_comp:10.2f} W")
print(f"  Q_L         {Q_L:10.2f} W")
print()
print("  energy balance check:  Q_L + W_comp - Q_H =",
      f"{Q_L + W_comp - Q_H:.6e} W   (must be ~0)")


> **Check your answer.** The ideal cycle gives **COP = 3.98559**.
> If you get something else, you changed a parameter. That is fine, but know
> which one.


## 5. The cycle on p–h axes

The plot is not decoration. The condenser is the top horizontal run, the
evaporator the bottom one, and the *width* of the bottom run is your
refrigeration effect. Widen it and the mass flow drops.


In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 5.0))

# saturation dome
Tc = am.critical(FLUID)["T"]
Ts = np.linspace(am.K(-30), Tc - 0.4, 300)
ax.plot([PropsSI("H","T",t,"Q",0,FLUID)/1e3 for t in Ts],
        [PropsSI("P","T",t,"Q",0,FLUID)/1e5 for t in Ts], color=am.MUTED, lw=1.3)
ax.plot([PropsSI("H","T",t,"Q",1,FLUID)/1e3 for t in Ts],
        [PropsSI("P","T",t,"Q",1,FLUID)/1e5 for t in Ts], color=am.MUTED, lw=1.3)

# the cycle, 1 -> 2 -> 3 -> 4 -> 1
hs = [h1, h2, h3, h4, h1]
ps = [p_evap, p_cond, p_cond, p_evap, p_evap]
ax.plot(np.array(hs)/1e3, np.array(ps)/1e5, "o-", color=am.ORANGE, lw=2.4, ms=7)
for n, (h, p) in enumerate(zip(hs[:4], ps[:4]), 1):
    ax.annotate(str(n), (h/1e3, p/1e5), textcoords="offset points",
                xytext=(9, 7), fontsize=12, color=am.NAVY, fontweight="bold")

ax.set_yscale("log")
ax.set_xlabel("specific enthalpy  h  (kJ/kg)")
ax.set_ylabel("pressure  p  (bar)")
ax.set_title(f"{FLUID} cycle:  {am.C(T_evap):.0f} °C to {am.C(T_cond):.0f} °C,"
             f"  COP = {COP_actual:.3f}")
ax.grid(alpha=.25, which="both")
plt.tight_layout(); plt.show()


## 6. The design question

The brief asks you to re-run with **`eta_s = 0.72`** and **`dT_sub = 5`**, and
report what happens to the COP and to the required mass flow.

Then answer this: **which of the two does a compressor salesman care about,
and why?**


In [ ]:
def cycle(eta_s=1.0, dT_sub=0.0, dT_sup=0.0, T_cond=T_cond, T_evap=T_evap):
    """The whole of sections 2-4, as one function. Returns a dict."""
    pe, pc = am.p_sat(FLUID, T_evap), am.p_sat(FLUID, T_cond)
    s1_ = (am.State(FLUID, P=pe, T=am.T_sat(FLUID, pe) + dT_sup) if dT_sup > 0
           else am.sat_vapour(FLUID, p=pe))
    h1_, s1v = s1_.h, s1_.s
    h2s_ = PropsSI("H", "P", pc, "S", s1v, FLUID)
    h2_  = h1_ + (h2s_ - h1_) / eta_s
    h3_  = (am.State(FLUID, P=pc, T=am.T_sat(FLUID, pc) - dT_sub).h if dT_sub > 0
            else am.sat_liquid(FLUID, p=pc).h)
    w, qh, ql = h2_ - h1_, h2_ - h3_, h1_ - h3_
    return {"COP": qh / w, "m_flow, kg/s": Q_H / qh, "W_comp, W": Q_H / qh * w,
            "T2, C": am.C(am.State(FLUID, P=pc, H=h2_).T),
            "q_L, kJ/kg": ql / 1e3}

base = cycle()
real = cycle(eta_s=0.72, dT_sub=5.0)

print(f"{'':16s}{'ideal':>12s}{'eta=0.72, dTsub=5':>20s}{'change':>12s}")
for k in base:
    a, b = base[k], real[k]
    print(f"{k:16s}{a:12.4f}{b:20.4f}{(b-a)/a*100:11.1f}%")


## 7. The deliverable

A sweep of condensing temperature, written to a filterable workbook. This is
what you hand in: **Python is the engine, Excel is the deliverable.**


In [ ]:
T_cond_C = np.arange(50, 81, 2.5)
rows = am.sweep(lambda t: cycle(eta_s=0.72, dT_sub=5.0, T_cond=am.K(t)),
                T_cond_C, name="T_cond, C")

path = am.to_excel(
    "AM5061_D1_HeatPump.xlsx",
    {"T_cond sweep": rows},
    title="AM5061 D-1 · 28 kW dairy heat pump · R134a",
    summary=[("Heating duty", Q_H, "W"),
             ("Refrigerant", FLUID, ""),
             ("Evaporating temperature", am.C(T_evap), "C"),
             ("Isentropic efficiency", 0.72, "-"),
             ("Subcooling", 5.0, "K"),
             ("COP at 70 C condensing", real["COP"], "-"),
             ("Mass flow at 70 C", real["m_flow, kg/s"], "kg/s")],
    sources=[("R134a properties", "CoolProp 8.0.0, Tillner-Roth & Baehr (1994) EOS"),
             ("Case specification", "AM5061 brief D-1, Jul-Nov 2026"),
             ("Cycle model", "Ideal vapour-compression, isenthalpic expansion")])

print("written:", path)
for r in rows[::3]:
    print(f"  T_cond {r['T_cond, C']:5.1f} C -> COP {r['COP']:.4f}, "
          f"m_flow {r['m_flow, kg/s']:.4f} kg/s")


Download the workbook from the file browser on the left (folder
icon), or run the cell below in Colab.


In [ ]:
try:
    from google.colab import files
    files.download("AM5061_D1_HeatPump.xlsx")
except ImportError:
    print("Not in Colab - the file is in this folder.")


## What to hand in

1. The four state points, as a table with units.
2. COP, mass flow and compressor power, for the ideal case and for
   `eta_s = 0.72, dT_sub = 5`.
3. The p–h diagram.
4. The workbook, with your sweep.
5. **One paragraph** answering the design question in section 6.

State your property source. A number without its source is unmarkable.
